# IOAI — 2024 First Stage Color Quantization (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/valid_data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-color-quantization/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 색상 양자화 — 모범답안 (비용인지 블렌드)

k-means 로 37색을 얻은 뒤, 각 팔레트 색을 **가장 가까운 단순색**(정육면체 꼭짓점 8색) 쪽으로 계수 α만큼 당긴다. 약간의 MSE 를 내주고 색 비용(가중치 21·42)을 크게 줄여 목적함수를 낮춘다(≈7290 → 100점). α로 MSE↔색비용을 절충.

## 데이터·유틸

In [ ]:
import numpy as np, glob
from PIL import Image
from sklearn.cluster import KMeans
np.random.seed(0)
val_files = sorted(glob.glob("data/valid_data/*.jpg"))
val = [np.array(Image.open(f).convert("RGB").resize((512,512)), dtype=np.uint8) for f in val_files]
print("valid images:", len(val))
V = np.array([[0,0,0],[0,0,255],[0,255,0],[0,255,255],[255,0,0],[255,0,255],[255,255,0],[255,255,255]], float)
def ensure37(colors):                      # 정확히 37개 서로 다른 uint8 색 보장
    seen=set(); out=[]
    for c in colors:
        c=np.clip(np.round(c),0,255).astype(int); t=tuple(c)
        while t in seen: c=(c+np.random.randint(-2,3,3)).clip(0,255); t=tuple(c)
        seen.add(t); out.append(c)
    return np.array(out)
def make37(img, colors, labels):
    C=ensure37(colors); q=C[labels].reshape(-1,3).copy()
    q[:37]=C[:37]                           # 37색 모두 등장하도록 심음 → 정확히 37 unique
    return q.reshape(img.shape).astype(np.uint8)

## your_quantization_algorithm — 비용인지 블렌드

In [ ]:
ALPHA = 0.5      # 팔레트 색을 가장 가까운 단순색 쪽으로 당기는 정도(MSE↔색비용 절충)
def your_quantization_algorithm(img):
    px = img.reshape(-1,3)
    km = KMeans(n_clusters=37, n_init=3, random_state=0).fit(px)
    C = km.cluster_centers_
    nearest = V[np.argmin(np.sqrt(((C[:,None]-V[None])**2).sum(2)), axis=1)]   # 각 색의 최근접 단순색
    C = (1-ALPHA)*C + ALPHA*nearest                                            # 단순색 쪽으로 블렌드
    return make37(img, C, km.labels_)

## 양자화 → submission.npz

In [ ]:
quant = np.stack([your_quantization_algorithm(img) for img in val]).astype(np.uint8)
np.savez_compressed("submission.npz", quantized=quant)
print("saved submission.npz", quant.shape)

α·색별 개별조정·목적함수 직접 최적화로 더 낮출 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)